# Session 4 — TPU: Static vs Dynamic Computation Graph

## Background

Sessions 1–3 used a fully static forward pass: the computation graph is identical on every step, which is what XLA requires to fuse operations and avoid recompilation. Real research often involves models with data-dependent control flow — sparse attention patterns, conditional skip connections, variable-length routing. On GPU, PyTorch's eager execution handles these without penalty. On TPU, a Python `if` statement whose condition depends on a tensor value forces XLA to either retrace and recompile the graph or fall back to slower interpreted execution. This session isolates that cost directly, using the same Transformer block as the baseline to make the comparison exact.

## Goal

At the end of this session, participants will understand why dynamic control flow is the primary constraint on TPU suitability for certain research workloads, will have measured the throughput penalty of a data-dependent branch on each device, and will be able to refactor a branching `forward()` into a compilable equivalent — and know when refactoring is worth the effort vs when the GPU is simply the right tool.

## Question

How much throughput does a Python branch in `forward()` cost on a compiled (TPU) vs eager (GPU) device, and does a tensor-valued equivalent recover it? Do realistic dynamic patterns (padding masks, early exit, variable-length loops) follow the same pattern?

---

**Runtime:** TPU (v5litepod-1 or equivalent)

After running, results are saved to `results/tpu_graph_variants.json` and loaded automatically by `03-analysis.ipynb`.

**Note:** `DynamicFF` forces XLA to sync on every step — expect it to be substantially slower than `StaticFF`. This is the expected result, not a bug.

In [1]:
import subprocess, sys, time
import torch, torch.nn as nn

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch==2.9.0", "torch_xla[tpu]==2.9.0",
        "-f", "https://storage.googleapis.com/libtpu-releases/index.html"
    ])
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr

device = torch_xla.device()

try:
    from torch_xla._internal import tpu as _tpu
    tpu_env = _tpu.get_tpu_env()
    chip    = tpu_env.get("ACCELERATOR_TYPE") or tpu_env.get("TPU_ACCELERATOR_TYPE", "unknown")
except Exception:
    chip = "unknown"

print(f"Python    : {sys.version.split()[0]}")
print(f"PyTorch   : {torch.__version__}")
print(f"torch_xla : {torch_xla.__version__}")
print(f"Device    : {device}")
print(f"TPU chip  : {chip}")

Python    : 3.12.12
PyTorch   : 2.9.0+cu128
torch_xla : 2.9.0
Device    : xla:0
TPU chip  : v5litepod-1


In [2]:
import sys; sys.path.insert(0, "..")
from transformer_block import BenchmarkTransformerBlock, D_MODEL, N_HEAD, DIM_FEEDFORWARD

# ── Session 4 config ──────────────────────────────────────────────────────────
BATCH_SIZE = 256
SEQ_LEN    = 128
N_STEPS, WARMUP = 50, 5

# ── Model variants ────────────────────────────────────────────────────────────

class StaticFF(nn.Module):
    """Baseline — identical to Sessions 1 and 2. Fully static graph."""
    def __init__(self):
        super().__init__()
        self.block = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)
    def forward(self, x):
        return self.block(x)


class DynamicFF(nn.Module):
    """Adds a Python if-else whose condition is a tensor value.
    On TPU: XLA must materialize out.mean() to CPU to evaluate the Python if,
    breaking lazy evaluation and forcing recompilation or interpreted fallback.
    """
    def __init__(self):
        super().__init__()
        self.block = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)
    def forward(self, x):
        out = self.block(x)
        if out.mean() > 0:   # Python branch on a tensor value — forces XLA sync
            return out
        else:
            return -out


class StaticEquivalentFF(nn.Module):
    """Replaces the Python branch with torch.where — a tensor op that stays in the XLA graph.
    Semantically equivalent to DynamicFF; XLA-compilable.
    """
    def __init__(self):
        super().__init__()
        self.block = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)
    def forward(self, x):
        out = self.block(x)
        return torch.where(out.mean() > 0, out, -out)  # condition stays in the graph


VARIANTS = {
    "StaticFF":           StaticFF,
    "DynamicFF":          DynamicFF,
    "StaticEquivalentFF": StaticEquivalentFF,
}

# ── Benchmark function ────────────────────────────────────────────────────────
def benchmark(model_class):
    model     = model_class().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    model.train()
    elapsed = 0.0
    for step in range(N_STEPS + WARMUP):
        x = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
        y = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
        torch_xla.sync()
        t0 = time.perf_counter()
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        torch_xla.sync()
        if step >= WARMUP:
            elapsed += time.perf_counter() - t0
    throughput = (N_STEPS * BATCH_SIZE) / elapsed
    latency_ms = (elapsed / N_STEPS) * 1000
    return throughput, latency_ms

print(f"Config  : batch={BATCH_SIZE}, seq_len={SEQ_LEN}, d_model={D_MODEL}")
print("Variants and benchmark function defined.")

Config  : batch=256, seq_len=128, d_model=512
Variants and benchmark function defined.


In [3]:
results = {}

for name, cls in VARIANTS.items():
    print(f"  [TPU] {name:<22} ... ", end="", flush=True)
    tput, lat = benchmark(cls)
    results[name] = {"throughput": round(tput, 1), "latency_ms": round(lat, 2)}
    print(f"{tput:,.0f} samples/sec  ({lat:.1f} ms/step)")

print("\nDone.")

# Show relative slowdown vs StaticFF baseline
baseline = results["StaticFF"]["throughput"]
print("\n── Relative throughput (vs StaticFF) ──────────────")
for name, r in results.items():
    ratio = r["throughput"] / baseline
    print(f"  {name:<22}: {ratio:.3f}×")

  [TPU] StaticFF               ... 

24,017 samples/sec  (10.7 ms/step)
  [TPU] DynamicFF              ... 

9,453 samples/sec  (27.1 ms/step)
  [TPU] StaticEquivalentFF     ... 

23,806 samples/sec  (10.8 ms/step)

Done.

── Relative throughput (vs StaticFF) ──────────────
  StaticFF              : 1.000×
  DynamicFF             : 0.394×
  StaticEquivalentFF    : 0.991×


---

## Realistic Scenarios  `# [PENDING RUN]`

The three abstract variants above isolate the pure overhead. The cells below benchmark realistic patterns. The TPU penalty should be most visible in Scenario 3 (variable loop count), which changes the XLA graph structure every step.

### Scenario 1 — Padding mask
Shape is constant — mask values vary. XLA can compile this statically.

### Scenario 2 — Conditional early exit
Python `if` on score forces sync; `torch.where` stays in-graph.

### Scenario 3 — Variable-length batch loop
Changing the loop count changes the XLA program — recompilation per unique count.

In [4]:
# [PENDING RUN] Scenario 1 — Padding mask
import torch.nn.functional as F

class PaddingMaskFF(nn.Module):
    """Boolean padding mask applied to attention — shape is static, values vary.
    XLA can compile this; mask values change each step but the graph does not.
    """
    def __init__(self):
        super().__init__()
        self.attn = nn.MultiheadAttention(D_MODEL, N_HEAD, batch_first=True)
        self.ff1  = nn.Linear(D_MODEL, DIM_FEEDFORWARD)
        self.ff2  = nn.Linear(DIM_FEEDFORWARD, D_MODEL)
        self.ln1  = nn.LayerNorm(D_MODEL)
        self.ln2  = nn.LayerNorm(D_MODEL)

    def forward(self, x, key_padding_mask):
        attn_out, _ = self.attn(x, x, x, key_padding_mask=key_padding_mask)
        x = self.ln1(x + attn_out)
        ff_out = self.ff2(F.gelu(self.ff1(x)))
        return self.ln2(x + ff_out)


def benchmark_padding_mask(n_steps=N_STEPS, warmup=WARMUP):
    model     = PaddingMaskFF().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    model.train()
    elapsed = 0.0
    for step in range(n_steps + warmup):
        seq_lens = torch.randint(64, SEQ_LEN + 1, (BATCH_SIZE,))
        mask = (torch.arange(SEQ_LEN).unsqueeze(0) >= seq_lens.unsqueeze(1)).to(device)
        x = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
        y = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
        torch_xla.sync()
        t0 = time.perf_counter()
        optimizer.zero_grad()
        loss = criterion(model(x, mask), y)
        loss.backward()
        optimizer.step()
        torch_xla.sync()
        if step >= warmup:
            elapsed += time.perf_counter() - t0
    return (n_steps * BATCH_SIZE) / elapsed, (elapsed / n_steps) * 1000


print("[TPU] Scenario 1 — Padding mask ... ", end="", flush=True)
tput, lat = benchmark_padding_mask()
results["PaddingMask"] = {"throughput": round(tput, 1), "latency_ms": round(lat, 2)}
print(f"{tput:,.0f} samples/sec  ({lat:.1f} ms/step)")

[TPU] Scenario 1 — Padding mask ... 

23,960 samples/sec  (10.7 ms/step)


In [5]:
# [PENDING RUN] Scenario 2 — Conditional early exit (Python if vs torch.where)

class EarlyExitDynamic(nn.Module):
    """Python if on score — forces XLA to materialize the scalar each step."""
    def __init__(self):
        super().__init__()
        self.block      = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)
        self.confidence = nn.Linear(D_MODEL, 1)
        self.threshold  = 0.5

    def forward(self, x):
        out   = self.block(x)
        score = torch.sigmoid(self.confidence(out[:, 0, :])).mean()
        if score.item() > self.threshold:   # forces XLA sync
            return out
        else:
            return self.block(out)


class EarlyExitStatic(nn.Module):
    """torch.where instead of Python if — XLA-compilable, no sync."""
    def __init__(self):
        super().__init__()
        self.block      = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)
        self.confidence = nn.Linear(D_MODEL, 1)
        self.threshold  = 0.5

    def forward(self, x):
        out    = self.block(x)
        score  = torch.sigmoid(self.confidence(out[:, 0, :])).mean()
        out2   = self.block(out)
        return torch.where(score > self.threshold, out, out2)


for name, cls in [("EarlyExitDynamic", EarlyExitDynamic), ("EarlyExitStatic", EarlyExitStatic)]:
    print(f"[TPU] Scenario 2 — {name} ... ", end="", flush=True)
    tput, lat = benchmark(cls)
    results[name] = {"throughput": round(tput, 1), "latency_ms": round(lat, 2)}
    print(f"{tput:,.0f} samples/sec  ({lat:.1f} ms/step)")

[TPU] Scenario 2 — EarlyExitDynamic ... 

7,522 samples/sec  (34.0 ms/step)
[TPU] Scenario 2 — EarlyExitStatic ... 

18,577 samples/sec  (13.8 ms/step)


In [6]:
# [PENDING RUN] Scenario 3 — Variable-length batch loop
# XLA is sensitive to loop iteration count: each unique count is a new graph.

N_PASSES_FIXED = 3

class FixedLoopFF(nn.Module):
    """Fixed number of passes — XLA traces once, reuses."""
    def __init__(self):
        super().__init__()
        self.block = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)

    def forward(self, x):
        for _ in range(N_PASSES_FIXED):
            x = self.block(x)
        return x


class VariableLoopFF(nn.Module):
    """Loop count derived from tensor value — forces XLA sync; different counts = different graphs."""
    def __init__(self):
        super().__init__()
        self.block = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)

    def forward(self, x, n_passes):
        for _ in range(n_passes):
            x = self.block(x)
        return x


def benchmark_variable_loop(n_steps=N_STEPS, warmup=WARMUP):
    model     = VariableLoopFF().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    model.train()
    elapsed = 0.0
    for step in range(n_steps + warmup):
        n_passes = 2 + (step % 3)   # cycles through 2, 3, 4 — three distinct XLA graphs
        x = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
        y = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
        torch_xla.sync()
        t0 = time.perf_counter()
        optimizer.zero_grad()
        loss = criterion(model(x, n_passes), y)
        loss.backward()
        optimizer.step()
        torch_xla.sync()
        if step >= warmup:
            elapsed += time.perf_counter() - t0
    return (n_steps * BATCH_SIZE) / elapsed, (elapsed / n_steps) * 1000


print("[TPU] Scenario 3 — FixedLoopFF     ... ", end="", flush=True)
tput, lat = benchmark(FixedLoopFF)
results["FixedLoopFF"] = {"throughput": round(tput, 1), "latency_ms": round(lat, 2)}
print(f"{tput:,.0f} samples/sec  ({lat:.1f} ms/step)")

print("[TPU] Scenario 3 — VariableLoopFF  ... ", end="", flush=True)
tput, lat = benchmark_variable_loop()
results["VariableLoopFF"] = {"throughput": round(tput, 1), "latency_ms": round(lat, 2)}
print(f"{tput:,.0f} samples/sec  ({lat:.1f} ms/step)")

[TPU] Scenario 3 — FixedLoopFF     ... 

15,565 samples/sec  (16.4 ms/step)
[TPU] Scenario 3 — VariableLoopFF  ... 

15,520 samples/sec  (16.5 ms/step)


In [7]:
import json, os
from datetime import datetime, timezone

def save_results(hardware, results, device_name="", batch_size=None, seq_len=None, results_dir="results"):
    os.makedirs(results_dir, exist_ok=True)
    payload = {
        "hardware":    hardware,
        "device_name": device_name,
        "session":     "4",
        "batch_size":  batch_size,
        "seq_len":     seq_len,
        "timestamp":   datetime.now(timezone.utc).isoformat(),
        "results":     results,
    }
    path = os.path.join(results_dir, f"{hardware.lower()}_graph_variants.json")
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    return path

path = save_results(
    "TPU", results,
    device_name=chip,
    batch_size=BATCH_SIZE,
    seq_len=SEQ_LEN,
    results_dir="results",
)
print(f"Saved → {path}")

Saved → results/tpu_graph_variants.json
